In [25]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

df2=pd.read_csv('/content/house_prices.csv')

print("DataFrame Head:")
df2.head()
print("\nDataFrame Info:")
df2.info()
print("\nDataFrame Description:")
df2.describe()

CSV_PATH = '/content/house_prices.csv'
RANDOM_STATE = 42

def load_data(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    print(f"Loaded {path} -> shape {df.shape}")
    return df


def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    before = len(df)
    df = df.drop_duplicates()
    if len(df) != before:
        print(f"Dropped {before - len(df)} duplicate rows")

    missing = df.isnull().sum()
    missing = missing[missing > 0]
    if len(missing):
        print(f"Filling missing values with column medians:\n{missing}")
        df = df.fillna(df.median(numeric_only=True))

    q1, q3 = df["price"].quantile([0.25, 0.75])
    iqr = q3 - q1
    upper = q3 + 3 * iqr
    n_outliers = (df["price"] > upper).sum()
    if n_outliers:
        print(f"Removing {n_outliers} extreme price outlier(s) above {upper:,.0f}")
        df = df[df["price"] <= upper]

    return df


def preprocess_splits(X_train: pd.DataFrame, X_test: pd.DataFrame):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    return X_train_scaled, X_test_scaled, scaler


def train_model(X_train, y_train) -> RandomForestRegressor:
    model = RandomForestRegressor(
        n_estimators=200, max_depth=10, random_state=RANDOM_STATE
    )
    model.fit(X_train, y_train)
    return model


def evaluate(model, X_train, y_train, X_test, y_test):
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    train_r2 = r2_score(y_train, train_pred)
    test_r2 = r2_score(y_test, test_pred)
    mae = mean_absolute_error(y_test, test_pred)
    rmse = np.sqrt(mean_squared_error(y_test, test_pred))

    print(f"\nTraining R^2 : {train_r2:.4f}")
    print(f"Test R^2     : {test_r2:.4f}")
    print(f"Test MAE     : {mae:,.0f}")
    print(f"Test RMSE    : {rmse:,.0f}")

    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring="r2")
    print(f"5-fold CV R^2: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

    return train_r2, test_r2


def predict_unseen_sample(model, scaler, feature_names):
    new_house = {
        "sqft_living": 2150,
        "bedrooms": 3,
        "bathrooms": 2.0,
        "floors": 2.0,
        "house_age": 8,
        "dist_to_city_km": 5.4,
        "quality_score": 7,
    }
    row = pd.DataFrame([new_house])[feature_names]
    row_scaled = scaler.transform(row)
    pred = model.predict(row_scaled)[0]

    print(f"\nUnseen sample input: {new_house}")
    print(f"Predicted price: ${pred:,.0f}")
    return pred


def main():
    df = load_data(CSV_PATH)
    df = clean_data(df)

    X = df.drop(columns="price")
    y = df["price"]
    feature_names = X.columns.tolist()

    X_train_raw, X_test_raw, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE
    )
    print(f"Train/test shapes: {X_train_raw.shape} / {X_test_raw.shape}")

    X_train, X_test, scaler = preprocess_splits(X_train_raw, X_test_raw)

    model = train_model(X_train, y_train)
    train_r2, test_r2 = evaluate(model, X_train, y_train, X_test, y_test)
    predict_unseen_sample(model, scaler, feature_names)

    print(f"\nFinal prediction score (test R^2): {test_r2:.4f}")
    return model, scaler, test_r2 # Return model and scaler


if __name__ == "__main__":
    global trained_model, trained_scaler # Declare as global
    trained_model, trained_scaler, final_r2 = main() # Assign to global variables

DataFrame Head:

DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   sqft_living      1000 non-null   int64  
 1   bedrooms         1000 non-null   int64  
 2   bathrooms        988 non-null    float64
 3   floors           1000 non-null   float64
 4   house_age        1000 non-null   int64  
 5   dist_to_city_km  1000 non-null   float64
 6   quality_score    1000 non-null   int64  
 7   price            1000 non-null   int64  
dtypes: float64(3), int64(5)
memory usage: 62.6 KB

DataFrame Description:
Loaded /content/house_prices.csv -> shape (1000, 8)
Filling missing values with column medians:
bathrooms    12
dtype: int64
Removing 3 extreme price outlier(s) above 939,799
Train/test shapes: (797, 7) / (200, 7)

Training R^2 : 0.9913
Test R^2     : 0.9499
Test MAE     : 20,442
Test RMSE    : 26,149
5-fold CV R^2: 0.9412 (+

In [21]:
data2=[2150, 3, 2.0, 2.0, 8, 5.4, 7]
print(data2)

[2150, 3, 2.0, 2.0, 8, 5.4, 7]


In [14]:
data_np=np.asarray(data2)
print(data_np)

[2150    3    2    0    2    8    5    4    7]


In [15]:
data_reshape=data_np.reshape(1,-1)
print(data_reshape)

[[2150    3    2    0    2    8    5    4    7]]


In [26]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='sklearn')
data_reshape = np.array(data2).reshape(1, -1)
input_std = trained_scaler.transform(data_reshape)
print(input_std)

[[ 0.39094725 -0.30087468 -0.54128261  0.62618435 -0.62438394 -0.34280516
   0.19144912]]


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [29]:
predicted_price = trained_model.predict(input_std)[0]
print(f"Predicted price for the new input: ${predicted_price:,.0f}")

Predicted price for the new input: $449,870


In [30]:
pred = trained_model.predict(input_std)[0]
print(f"Predicted price for the new input: ${pred:,.0f}")

Predicted price for the new input: $449,870
